# Bio_ClinicalBERT Fine-Tuning Pipeline for Clinical NER

> **Dataset**: `SYN-CLINNER-v1` (Synthetic Clinical Named Entity Recognition dataset)
> **Model**: `emilyalsentzer/Bio_ClinicalBERT`
> **Target Hardware**: NVIDIA GPU (Google Colab T4 / V100 / A100)
> **Storage**: Persistent model checkpoints and reports saved to Google Drive

---
### Pipeline Objectives
1. Check that an NVIDIA GPU is available (CUDA).
2. Install required dependencies (`transformers`, `accelerate`, `seqeval`).
3. Mount Google Drive for persistent checkpoint and artifact storage.
4. Verify the `SYN-CLINNER-v1` dataset splits (350 train / 75 val / 75 test).
5. Verify all 19 deterministic BIO labels.
6. Run a fast tokenizer preprocessing smoke test.
7. Train Bio_ClinicalBERT token classification using Hugging Face Trainer and AdamW.
8. Evaluate on validation after each epoch, saving the best checkpoint based on validation entity-level F1.
9. Evaluate on the held-out test set only after training and model selection are complete.
10. Save best model weights, tokenizer, config, and comprehensive evaluation reports to Google Drive.

## Step 1: Check NVIDIA GPU Availability
Confirm that a GPU runtime is active in Colab (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
    print(f"Compute Cap.:    {torch.cuda.get_device_capability(0)}")
else:
    raise RuntimeError("No GPU detected! Please enable GPU via Runtime > Change runtime type.")

!nvidia-smi

## Step 2: Install Dependencies
Install `transformers`, `accelerate`, and `seqeval`.

In [ ]:
!pip install -q "transformers>=4.40.0" "accelerate>=0.30.0" "seqeval>=1.2.2"

## Step 3: Mount Google Drive
Mount Google Drive for persistent checkpoint and report storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 4: Configure Project Path & Source Code
Set the repository root path and add `src/` to `sys.path`.

In [ ]:
import os
import sys
from pathlib import Path

# If repository is cloned locally in /content/clinical-entity-extraction:
LOCAL_REPO = Path('/content/clinical-entity-extraction')
# If repository is stored directly in Google Drive:
DRIVE_REPO = Path('/content/drive/MyDrive/clinical-entity-extraction')

if LOCAL_REPO.exists():
    PROJECT_ROOT = LOCAL_REPO
elif DRIVE_REPO.exists():
    PROJECT_ROOT = DRIVE_REPO
else:
    PROJECT_ROOT = Path(os.getcwd())

print(f"Project root configured: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"sys.path includes: {SRC_DIR}")

## Step 5: Verify Dataset Files & Record Counts
Verify `train.jsonl` (350), `validation.jsonl` (75), and `test.jsonl` (75).

In [ ]:
from preprocessing import load_jsonl, validate_records

train_path = PROJECT_ROOT / 'datasets/syn_clinner_v1/data/train.jsonl'
val_path = PROJECT_ROOT / 'datasets/syn_clinner_v1/data/validation.jsonl'
test_path = PROJECT_ROOT / 'datasets/syn_clinner_v1/data/test.jsonl'

train_recs = load_jsonl(train_path)
val_recs = load_jsonl(val_path)
test_recs = load_jsonl(test_path)

validate_records(train_recs)
validate_records(val_recs)
validate_records(test_recs)

print("✓ Dataset integrity verified:")
print(f"  Train records:      {len(train_recs):3d} (expected 350)")
print(f"  Validation records: {len(val_recs):3d} (expected  75)")
print(f"  Test records:       {len(test_recs):3d} (expected  75)")
print(f"  Total records:      {len(train_recs) + len(val_recs) + len(test_recs):3d} (expected 500)")

assert len(train_recs) == 350
assert len(val_recs) == 75
assert len(test_recs) == 75

## Step 6: Verify 19 BIO Labels & Mapping
Verify the exact 19 deterministic BIO labels derived from the 9 entity categories.

In [ ]:
from preprocessing import build_label_maps, ENTITY_TYPES

labels, label2id, id2label = build_label_maps()
print(f"Total unique labels: {len(labels)} (expected 19)")
print(f"Entity categories ({len(ENTITY_TYPES)}): {list(ENTITY_TYPES)}\n")
print("BIO Label ID mapping:")
for idx in range(len(labels)):
    print(f"  {idx:2d} -> {id2label[idx]}")

assert len(labels) == 19
assert id2label[0] == "O"

## Step 7: Preprocessing & Fast Tokenizer Smoke Test
Smoke test tokenization and BIO alignment on sample records from each split.

In [ ]:
from transformers import AutoTokenizer
from preprocessing import encode_record, MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

for split_name, rec in [("train", train_recs[0]), ("validation", val_recs[0]), ("test", test_recs[0])]:
    enc = encode_record(rec, tokenizer=tokenizer, label2id=label2id, max_length=512)
    non_o = [id2label[lid] for lid in enc.label_ids if lid not in (-100, 0)]
    print(f"✓ {split_name:10s} [{rec['id']}]: {len(enc.input_ids):3d} tokens, "
          f"{len(enc.assertions):2d} entities, tagged tokens: {len(non_o)}")

print("\nSmoke test passed successfully.")

## Step 8: Start Fine-Tuning Bio_ClinicalBERT with CUDA
Launch fine-tuning with suggested initial hyperparameters:
- `max_length`: 512
- `batch_size`: 8
- `epochs`: 5
- `learning_rate`: 2e-5
- `weight_decay`: 0.01
- `warmup_ratio`: 0.1
- `seed`: 42
- `early_stopping_patience`: 3
- `output_dir`: Persistent path in Google Drive

> Note: The held-out test set is evaluated ONLY after training and model selection are complete.

In [ ]:
from train import TrainConfig, train_model

# Persistent Google Drive checkpoint directory
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/clinicalbert_checkpoints/syn_clinner_v1"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

config = TrainConfig(
    train_file=str(train_path),
    val_file=str(val_path),
    test_file=str(test_path),
    output_dir=DRIVE_OUTPUT_DIR,
    max_length=512,
    batch_size=8,
    epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    seed=42,
    early_stopping_patience=3,
    save_total_limit=2,
    logging_steps=10,
    do_test=True,
)

print("=== Launching Fine-Tuning on CUDA ===")
print(f"Target Output Directory: {config.output_dir}")
training_metadata = train_model(config)

## Step 9: Held-Out Test Set Evaluation Results
Review final test set metrics, per-class F1, and error analysis generated by `src/evaluator.py`.

In [ ]:
import json
from pathlib import Path

report_file = Path(DRIVE_OUTPUT_DIR) / "test_report.json"

if report_file.exists():
    with report_file.open("r", encoding="utf-8") as fh:
        test_results = json.load(fh)

    print("=" * 60)
    print("        HELD-OUT TEST SET EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Overall Precision: {test_results['precision']:.4f}")
    print(f"Overall Recall:    {test_results['recall']:.4f}")
    print(f"Overall F1 Score:  {test_results['f1']:.4f}")
    print("-" * 60)
    print("\nClassification Report (per entity category):")
    print(test_results["classification_report"])
    print("-" * 60)
    print("\nError Analysis Summary:")
    for k, v in test_results["error_analysis"]["summary"].items():
        print(f"  {k:25s}: {v}")
    print("=" * 60)
else:
    print(f"Report file {report_file} not found.")

## Step 10: Persistent Checkpoint & Artifacts Summary
List all saved artifacts in Google Drive (model weights, tokenizer, configs, and reports).

In [ ]:
checkpoint_files = sorted(Path(DRIVE_OUTPUT_DIR).glob("*"))
print(f"=== Final Saved Artifacts in Google Drive ({len(checkpoint_files)} items) ===")
print(f"Location: {DRIVE_OUTPUT_DIR}\n")
for f in checkpoint_files:
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name:<35} ({size_mb:8.2f} MB)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}")